# LABORATÓRIO 10: O Pipeline Definitivo
## RAG, QLoRA e Otimização de Inferência na GPU

> **Partes deste laboratório foram geradas/complementadas com IA, revisadas e validadas por Heitor Viana**

---

### Estrutura dos Commits
- **Commit 1** — Setup do ambiente e carregamento do modelo em 4-bit (Passos 1 e 2)
- **Commit 2** — Benchmark baseline: geração sem KV Cache (Passo 3)
- **Commit 3** — Otimização: KV Cache + SDPA fallback + análise arquitetural (Passos 4 e 5)

---

## ⚠️ Nota sobre FlashAttention-2
FlashAttention-2 exige GPU **Ampere ou superior** (A100, RTX 3090+).  
A **Tesla T4** do Colab free (Turing, sm_75) **não suporta** FA2.  
Este notebook detecta automaticamente o suporte e aplica **fallback para `sdpa` (Scaled Dot-Product Attention)** do PyTorch ≥ 2.0, que usa `F.scaled_dot_product_attention` com kernels otimizados e produz resultados equivalentes em qualquer GPU.

---
# COMMIT 1 — Setup do Ambiente e Ingestão Eficiente

In [ ]:
# ── Célula 1: Instalação das dependências ──────────────────────────────────────
# Execute apenas uma vez por sessão de runtime.

!pip install -q \
    "transformers>=4.40.0" \
    "bitsandbytes>=0.43.0" \
    "accelerate>=0.29.0" \
    "torch>=2.1.0"

# Instala flash-attn apenas quando a GPU tem suporte real a FlashAttention-2.
import subprocess, sys
import torch

if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "flash-attn", "--no-build-isolation"],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print("⚠️  flash-attn não instalado; usaremos SDPA como fallback.")
    else:
        print("✅ flash-attn instalado com sucesso.")
else:
    print("ℹ️  GPU sem suporte a FlashAttention-2 (ex.: T4) ou sem CUDA; usaremos SDPA/eager.")

In [ ]:
# ── Célula 2: Imports e verificação de hardware ────────────────────────────────
import torch
import time
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ── Detecção de GPU e suporte a FlashAttention-2 ──────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def detect_attn_implementation() -> str:
    """
    Retorna a melhor implementação de atenção disponível:
      - 'flash_attention_2' : GPU Ampere+ (sm >= 80) com flash-attn instalado
      - 'sdpa'              : PyTorch Scaled Dot-Product Attention (fallback universal)
      - 'eager'             : implementação padrão (CPU ou GPU muito antiga)
    """
    if DEVICE != "cuda":
        print("ℹ️  Sem GPU — usando implementação eager (CPU).")
        return "eager"

    major, minor = torch.cuda.get_device_capability()
    gpu_name  = torch.cuda.get_device_name(0)
    print(f"🖥️  GPU detectada : {gpu_name}  |  Compute Capability: sm_{major}{minor}")

    if major >= 8:  # Ampere (A100/A10/RTX 30xx) ou superior
        try:
            import flash_attn  # noqa: F401
            print("✅ FlashAttention-2 disponível e ativado.")
            return "flash_attention_2"
        except ImportError:
            print("⚠️  flash-attn não encontrado — caindo para SDPA.")

    # Turing (T4, sm_75) ou Volta (sm_70): SDPA via PyTorch
    if hasattr(torch.nn.functional, "scaled_dot_product_attention"):
        print("🔄 Usando PyTorch SDPA (Scaled Dot-Product Attention) como fallback.")
        return "sdpa"

    print("⚠️  SDPA não disponível — usando eager attention.")
    return "eager"

ATTN_IMPL = detect_attn_implementation()
print(f"\n📌 Implementação de atenção selecionada: {ATTN_IMPL}")

In [ ]:
# ── Célula 3: Configuração QLoRA 4-bit e carregamento do modelo ────────────────
#
# PASSO 1 — Ingestão Eficiente (Revisando a Unidade II)
# Carregamos o LLM em NF4 (4-bit) para caber na VRAM disponível.
# Sem quantização, TinyLlama-1.1B em float16 ocupa ~2.2 GB;
# com QLoRA 4-bit o footprint cai para ~0.7 GB.

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",             # NormalFloat4 — padrão QLoRA
    bnb_4bit_compute_dtype=torch.float16,  # Operações em fp16 para velocidade
    bnb_4bit_use_double_quant=True,        # Double quantization: economiza ~0.4 bpw extra
)

print(f"⏳ Carregando {MODEL_ID} em 4-bit (NF4)...")

# Zeramos o contador de memória antes do carregamento
if DEVICE == "cuda":
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",                     # Distribui automaticamente na GPU disponível
    attn_implementation=ATTN_IMPL,         # SDPA ou flash_attention_2 conforme detecção
    torch_dtype=torch.float16,
)
model.eval()

# ── Métrica do Passo 1: VRAM após carregamento ────────────────────────────────
if DEVICE == "cuda":
    torch.cuda.synchronize()
    vram_model_mb = torch.cuda.memory_allocated() / 1024**2
    vram_peak_mb  = torch.cuda.max_memory_allocated() / 1024**2
    print(f"\n📊 MÉTRICA — VRAM após carregamento do modelo:")
    print(f"   Alocado atual : {vram_model_mb:.1f} MB")
    print(f"   Pico até agora: {vram_peak_mb:.1f} MB")
    print(f"   Implementação : {ATTN_IMPL}")
else:
    print("\n⚠️  Sem GPU: métricas de VRAM indisponíveis.")

In [ ]:
# ── Célula 4: Simulação do RAG massivo ────────────────────────────────────────
#
# PASSO 2 — Simulando o RAG Massivo
# Geramos um contexto fictício de ~12.000 tokens imitando 5 capítulos
# de manuais médicos recuperados pelo banco vetorial do Lab 09.

def gerar_contexto_medico_ficticio(n_capitulos: int = 5) -> str:
    """Gera texto fictício simulando manuais médicos recuperados pelo RAG."""

    capitulos = []
    temas = [
        ("Farmacologia Clínica",
         "Os inibidores da ECA (enzima conversora de angiotensina) constituem a pedra angular "
         "do tratamento da hipertensão arterial sistêmica e da insuficiência cardíaca congestiva. "
         "Seu mecanismo de ação envolve o bloqueio competitivo da enzima responsável pela conversão "
         "da angiotensina I em angiotensina II, reduzindo assim a vasoconstrição periférica e a "
         "retenção de sódio mediada pela aldosterona. Entre os representantes desta classe destacam-se "
         "o enalapril, o lisinopril, o ramipril e o captopril, cada qual com perfil farmacocinético "
         "distinto. O enalapril, pró-fármaco, é convertido no fígado em enalaprilato, sua forma ativa. "
         "Os efeitos adversos mais relevantes incluem tosse seca persistente (incidência de 10–15%), "
         "hipotensão de primeira dose e, raramente, angioedema potencialmente fatal. "),

        ("Fisiologia Cardiovascular",
         "O ciclo cardíaco compreende duas fases principais: sístole e diástole. Durante a sístole "
         "ventricular, a pressão intracavitária supera a pressão aórtica, abrindo a valva aórtica e "
         "ejetando aproximadamente 70 mL de sangue (volume sistólico). O débito cardíaco é o produto "
         "da frequência cardíaca pelo volume sistólico, com valores de repouso em torno de 5 L/min. "
         "A lei de Frank-Starling estabelece que o aumento do volume diastólico final resulta em maior "
         "força de contração, até o limite da capacidade de estiramento miocárdico. A fração de ejeção, "
         "calculada como (VDF − VSF)/VDF, é o marcador clínico de função sistólica mais empregado, "
         "sendo considerada preservada acima de 50%. "),

        ("Semiologia Médica",
         "A ausculta cardíaca deve ser realizada em ambiente silencioso com o paciente em decúbito "
         "dorsal, posição sentada e decúbito lateral esquerdo. As quatro áreas clássicas de ausculta são: "
         "aórtica (2° EID), pulmonar (2° EIE), tricúspide (4° EIE junto ao esterno) e mitral (5° EIE "
         "na linha hemiclavicular). A primeira bulha (B1) corresponde ao fechamento das valvas "
         "atrioventriculares e marca o início da sístole; a segunda bulha (B2) marca o fechamento das "
         "valvas semilunares e o início da diástole. Sopros holosistólicos de regurgitação mitral "
         "irradiam para a axila, enquanto sopros de estenose aórtica irradiam para os vasos do pescoço. "),

        ("Pneumologia",
         "A DPOC (Doença Pulmonar Obstrutiva Crônica) é caracterizada por limitação persistente e "
         "progressiva ao fluxo aéreo, geralmente associada ao tabagismo crônico. O diagnóstico é "
         "espirométrico: VEF1/CVF < 0,70 pós-broncodilatador confirma a obstrução. A classificação "
         "GOLD estratifica a gravidade em estágios I a IV com base no VEF1 percentual previsto. "
         "O tratamento inclui cessação tabágica, broncodilatadores de longa duração (LABA e LAMA), "
         "corticosteroides inalatórios nas formas graves e oxigenioterapia domiciliar quando PaO2 "
         "< 55 mmHg. As exacerbações agudas — geralmente infecciosas — aumentam a morbimortalidade "
         "e aceleram o declínio da função pulmonar. "),

        ("Endocrinologia e Diabetes",
         "O diabetes mellitus tipo 2 resulta da combinação de resistência periférica à insulina e "
         "disfunção progressiva das células beta pancreáticas. O diagnóstico é estabelecido por glicemia "
         "de jejum ≥ 126 mg/dL, glicemia 2h pós-TOTG ≥ 200 mg/dL ou HbA1c ≥ 6,5% em duas ocasiões. "
         "A metformina permanece como primeira linha terapêutica, com mecanismo primário de inibição "
         "da gliconeogênese hepática. Inibidores de SGLT-2 (empagliflozina, dapagliflozina) e análogos "
         "de GLP-1 (semaglutida, liraglutida) demonstraram benefícios cardiovasculares e renais "
         "independentes do controle glicêmico em ensaios clínicos de desfechos cardiovasculares. "),
    ]

    for i, (titulo, paragrafo_base) in enumerate(temas[:n_capitulos]):
        # Cada capítulo repete e expande o parágrafo base para atingir ~2.400 tokens
        conteudo = (
            f"CAPÍTULO {i+1}: {titulo}\n\n"
            + (paragrafo_base * 18)   # ~2.400 tokens por capítulo
            + "\n\n"
            + "REFERÊNCIAS: Diretrizes da Sociedade Brasileira de Cardiologia (2023); "
            + "Harrison's Principles of Internal Medicine, 21ª ed.; "
            + "UpToDate Clinical Decision Support, acesso 2024.\n"
            + "─" * 80 + "\n"
        )
        capitulos.append(conteudo)

    return "\n".join(capitulos)


# Gera o contexto e tokeniza
contexto_medico = gerar_contexto_medico_ficticio(n_capitulos=5)

# Monta o prompt no formato chat do TinyLlama
PROMPT_SISTEMA = (
    "Você é um assistente médico especializado em resumos clínicos. "
    "Analise o contexto fornecido e produza um resumo clínico objetivo."
)

PROMPT_USUARIO = (
    f"Contexto recuperado pelo sistema RAG:\n\n{contexto_medico}\n\n"
    "Com base exclusivamente no contexto acima, escreva um resumo clínico "
    "de 500 palavras abordando os principais pontos de cada capítulo."
)

messages = [
    {"role": "system", "content": PROMPT_SISTEMA},
    {"role": "user",   "content": PROMPT_USUARIO},
]

prompt_formatado = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

input_ids = tokenizer(
    prompt_formatado,
    return_tensors="pt",
    truncation=True,
    max_length=4096,      # Limite de contexto do TinyLlama
).input_ids.to(DEVICE)

n_tokens_contexto = input_ids.shape[1]
print(f"📊 MÉTRICA — Contexto tokenizado:")
print(f"   Texto bruto    : {len(contexto_medico):,} caracteres")
print(f"   Tokens no prompt (após truncamento): {n_tokens_contexto:,}")
print(f"\n✅ Commit 1 concluído: modelo carregado + contexto RAG simulado.")

---
# COMMIT 2 — Benchmark Baseline: Geração SEM KV Cache

**Passo 3 — O Gargalo de Geração (O Problema do Decoder)**

Com `use_cache=False`, a cada novo token gerado o modelo precisa recomputar os vetores **Q, K, V** para *todos* os tokens anteriores.  
Isso causa complexidade **O(n²)** em tempo e memória, onde **n** é o comprimento da sequência.

In [ ]:
# ── Célula 5: Geração SEM KV Cache (baseline lento) ───────────────────────────
N_NOVOS_TOKENS = 100

def gerar_sem_cache(model, input_ids, n_tokens: int) -> dict:
    """
    Geração token-a-token SEM KV Cache.
    Simula o pior caso: recálculo completo de Q, K, V a cada passo.
    """
    model.config.use_cache = False   # ← A pegadinha do Passo 3

    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

    t_inicio = time.perf_counter()

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=n_tokens,
            do_sample=False,          # Greedy decoding para reprodutibilidade
            use_cache=False,          # Explicitamente desabilitado
            pad_token_id=tokenizer.eos_token_id,
        )

    if DEVICE == "cuda":
        torch.cuda.synchronize()

    t_total = time.perf_counter() - t_inicio

    vram_pico_mb = (
        torch.cuda.max_memory_allocated() / 1024**2
        if DEVICE == "cuda" else 0.0
    )

    tokens_gerados = output.shape[1] - input_ids.shape[1]
    tokens_por_seg = tokens_gerados / t_total if t_total > 0 else 0

    return {
        "tempo_s"       : t_total,
        "tokens_gerados": tokens_gerados,
        "tok_por_seg"   : tokens_por_seg,
        "vram_pico_mb"  : vram_pico_mb,
        "output"        : output,
    }


print("⏳ Gerando SEM KV Cache... (pode demorar bastante)")
metricas_sem_cache = gerar_sem_cache(model, input_ids, N_NOVOS_TOKENS)

print(f"\n📊 MÉTRICAS — Geração SEM KV Cache (baseline):")
print(f"   Tokens gerados    : {metricas_sem_cache['tokens_gerados']}")
print(f"   Tempo total       : {metricas_sem_cache['tempo_s']:.2f} s")
print(f"   Throughput        : {metricas_sem_cache['tok_por_seg']:.2f} tokens/s")
print(f"   Pico VRAM         : {metricas_sem_cache['vram_pico_mb']:.1f} MB")
print(f"   Implementação att.: {ATTN_IMPL}")

texto_sem_cache = tokenizer.decode(
    metricas_sem_cache["output"][0][input_ids.shape[1]:],
    skip_special_tokens=True
)
print(f"\n📝 Saída (primeiros 300 chars):\n{texto_sem_cache[:300]}...")
print(f"\n✅ Commit 2 concluído: benchmark sem cache registrado.")

---
# COMMIT 3 — Otimização: KV Cache + SDPA/FlashAttention-2

**Passo 4 — A Engenharia de Otimização**

| Técnica | O quê resolve | Como funciona |
|---|---|---|
| **KV Cache** | Recálculo redundante de K e V | Armazena os tensores K/V de tokens já processados na SRAM |
| **SDPA / FA2** | Materialização O(n²) da matriz de atenção na VRAM | Computa atenção em blocos, nunca materializando a matriz completa |
| **QLoRA 4-bit** | Footprint estático do modelo | Reduz pesos de 16-bit para 4-bit com quantização NF4 |

In [ ]:
# ── Célula 6: Geração COM KV Cache (otimizado) ────────────────────────────────
# O modelo já foi carregado com SDPA ou FA2 na Célula 3.
# Aqui apenas reativamos o use_cache e comparamos as métricas.

def gerar_com_cache(model, input_ids, n_tokens: int) -> dict:
    """
    Geração COM KV Cache ativado.
    K e V dos tokens do prompt são calculados uma única vez e reutilizados
    a cada passo de decodificação — complexidade temporal cai para O(n).
    """
    model.config.use_cache = True   # ← Reativa o KV Cache

    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

    t_inicio = time.perf_counter()

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=n_tokens,
            do_sample=False,
            use_cache=True,          # KV Cache ativado
            pad_token_id=tokenizer.eos_token_id,
        )

    if DEVICE == "cuda":
        torch.cuda.synchronize()

    t_total = time.perf_counter() - t_inicio

    vram_pico_mb = (
        torch.cuda.max_memory_allocated() / 1024**2
        if DEVICE == "cuda" else 0.0
    )

    tokens_gerados = output.shape[1] - input_ids.shape[1]
    tokens_por_seg = tokens_gerados / t_total if t_total > 0 else 0

    return {
        "tempo_s"       : t_total,
        "tokens_gerados": tokens_gerados,
        "tok_por_seg"   : tokens_por_seg,
        "vram_pico_mb"  : vram_pico_mb,
        "output"        : output,
    }


print("⏳ Gerando COM KV Cache + atenção otimizada...")
metricas_com_cache = gerar_com_cache(model, input_ids, N_NOVOS_TOKENS)

print(f"\n📊 MÉTRICAS — Geração COM KV Cache + {ATTN_IMPL}:")
print(f"   Tokens gerados    : {metricas_com_cache['tokens_gerados']}")
print(f"   Tempo total       : {metricas_com_cache['tempo_s']:.2f} s")
print(f"   Throughput        : {metricas_com_cache['tok_por_seg']:.2f} tokens/s")
print(f"   Pico VRAM         : {metricas_com_cache['vram_pico_mb']:.1f} MB")
print(f"   Implementação att.: {ATTN_IMPL}")

texto_com_cache = tokenizer.decode(
    metricas_com_cache["output"][0][input_ids.shape[1]:],
    skip_special_tokens=True
)
print(f"\n📝 Saída (primeiros 300 chars):\n{texto_com_cache[:300]}...")

In [ ]:
# ── Célula 7: Tabela comparativa de métricas ──────────────────────────────────
import math

def speedup(base, otim):
    return base / otim if otim > 0 else math.inf

def reducao_pct(base, otim):
    return (1 - otim / base) * 100 if base > 0 else 0

print("\n" + "═" * 60)
print("   RESUMO COMPARATIVO DE MÉTRICAS — LAB 10")
print("═" * 60)
print(f"{'Métrica':<28} {'Sem Cache':>12} {'Com Cache':>12} {'Δ':>10}")
print("─" * 60)
print(
    f"{'Tempo de geração (s)':<28} "
    f"{metricas_sem_cache['tempo_s']:>11.2f}s "
    f"{metricas_com_cache['tempo_s']:>11.2f}s "
    f"{speedup(metricas_sem_cache['tempo_s'], metricas_com_cache['tempo_s']):>8.1f}x"
)
print(
    f"{'Throughput (tok/s)':<28} "
    f"{metricas_sem_cache['tok_por_seg']:>10.2f}   "
    f"{metricas_com_cache['tok_por_seg']:>10.2f}   "
    f"+{metricas_com_cache['tok_por_seg'] - metricas_sem_cache['tok_por_seg']:>6.1f}"
)
print(
    f"{'Pico VRAM (MB)':<28} "
    f"{metricas_sem_cache['vram_pico_mb']:>10.1f}   "
    f"{metricas_com_cache['vram_pico_mb']:>10.1f}   "
    f"-{reducao_pct(metricas_sem_cache['vram_pico_mb'], metricas_com_cache['vram_pico_mb']):>5.1f}%"
)
print("═" * 60)
print(f"Implementação de atenção: {ATTN_IMPL}")
print("\n✅ Commit 3 concluído: otimização implementada e métricas comparadas.")

In [ ]:
# ── Célula 8: Limpeza de memória (boa prática ao final) ───────────────────────
del model
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()
    print("🗑️  VRAM liberada.")
print("🏁 Laboratório 10 finalizado.")